In [4]:
import os
import json
import re
import time
import random
from mistralai import Mistral
from collections import defaultdict
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
from owlready2 import *


In [5]:
api_key = "ujI60UR6Fe5jel48SAtfnMiN5Skxfwhq"
model =  "open-mistral-nemo"
client = Mistral(api_key=api_key)
SEED = 69
TEMPERATURE = 0.7
clustered_questions_path = r"competency_questions_output/clustered_CQs_with_promoted_and_reviewed_noise.json"
output_file_path = "ontology_output/ontology_fabi_experimet.json"


In [6]:
#Load data
with open(clustered_questions_path, "r", encoding="utf-8") as f:
    data = json.load(f)


In [7]:
base_system_prompt = """
Du bist ein professioneller Ontologie-Extraktor, der auf juristische Texte spezialisiert ist.
Deien Antwort muss in JSON-Format sein, ohne zusätzliche Erklärungen oder Kommentare.

Ihre Aufgabe ist es, Ontologiekomponenten aus den Wettbewerbsabfragen zu extrahieren, die aus (deutschen Gerichtsentscheidungen) generiert wurden. 
Identifizieren Sie immer drei strukturierte Komponenten und geben Sie diese zurück:

1. **Klassen**: Rechtsträger oder Begriffe (z.B. Urteil, Entscheidungsgrund, Anspruch, Person, Tatbestand, Eiwendung, Rubrum, Tenor)
2. **Eigenschaften**: Verben oder Phrasen, die Beziehungen oder Eigenschaften darstellen (z. B. beinhaltet, basiert auf, verhindert, erörtern)
3. **Beziehungen**: Tripel (Subjekt)-(Prädikat)-(Objekt), die zwei Klassen über eine Eigenschaft miteinander verbinden

Anforderungen:
- Extrahieren Sie Begriffe nur, wenn sie Beziehungen haben.
- Klare juristische Sprache in Deutsch verwenden.
- Die Ausgabe muss dieser JSON-Struktur folgen (Beispiel):

{
    "Klassen": ["Klasse1", "Klasse2"],
    "Eigenschaften": ["Eigenschaft1", "Eigenschaft2"],
    "Beziehungen": [
        ["Urteil", "beinhaltet", "Tatbestand"],
        ["Anspruch", "basiert auf", "Tatbestand"]
    ]
}

## Modellierungsrichtlinien
- **Konzept Identifikation**:
- Identifizieren Sie Substantive und Substantivphrasen als potenzielle **Klassen**.
- Verben und Verbphrasen als mögliche **Eigenschaften** identifizieren.
- Präpositionen identifizieren, die Beziehungen zwischen Substantiven herstellen.

- **Klassen**:
- Klassen müssen immer im Singular stehen.
- Repräsentieren Konzepte in der Domäne, nicht die Wörter, die diese Konzepte bezeichnen.
- Vermeiden Sie die Bildung von Klassen für Synonyme; verwenden Sie eine einzige Klasse für Konzepte mit der gleichen Bedeutung.

- **Eigenschaften**:
- Stellen wichtige Beziehungen oder Attribute zwischen Klassen dar.
- Sollten aussagekräftig sein und eine signifikante Verbindung darstellen.

- **Beziehungen**:
- Stellen Sie Verbindungen zwischen Klassen mithilfe von Eigenschaften her.
- Sicherstellen, dass Beziehungen innerhalb des Domänenkontexts sinnvoll sind.

- **Allgemeine Grundsätze**:
- Es gibt keinen einzigen richtigen Weg, eine Domäne zu modellieren; die beste Lösung hängt von der Anwendung ab.
- Ontologieentwicklung ist ein iterativer Prozess; Verfeinerung nach Bedarf.
- Vermeiden Sie Zyklen in der Klassenhierarchie.
- Geschwister in der Hierarchie sollten auf der gleichen Ebene der Allgemeinheit sein.


## Namensgebung
**Allgemein**:
- Fügen Sie keine Zeichenfolgen wie 'class', 'domain', 'range', 'property' oder 'slot' zu Namen hinzu.
- Verwenden Sie eine konsistente Namensgebung in der gesamten Ontologie.

**Klassen**:
- Namen werden immer großgeschrieben.
- Verwenden Sie Substantive oder zusammengesetzte Substantive (z. B. Vertrag, Arbeit, Arbeitsvertrag).
- Immer Singular statt Plural verwenden (z. B. Arbeitsvertrag statt Arbeitsverträge).
- Vermeiden Sie Abkürzungen (z.B. Arbeitgeber statt AG).

**Eigenschaften**:
- Namen beginnen mit einem Kleinbuchstaben.
- Verwenden Sie Verben oder Verbphrasen.
- Kann Substantive in CamelCase enthalten, die mit einem Verb beginnen (z. B. hatAnspruchAuf).
- Keine Leerzeichen, Kommas, Sternchen oder Sonderzeichen enthalten."


##Finally
Stellen Sie sicher, dass Sie leere Listen für Klassen, Eigenschaften oder Beziehungen einfügen, wenn keine gefunden werden.
Stellen Sie sicher, dass alle extrahierten Komponenten, d.h. Klassen, Eigenschaften und Beziehungen sowie alle Beschreibungen auf Deutsch sind und übersetzen Sie, wo nötig.
Nehmen Sie eine Klasse nur auf, wenn mindestens eine Beziehung für eine Klasse gefunden wird. Überprüfen Sie diese Anforderung.
Prüfen Sie sorgfältig für jede Klasse ohne Beziehung, basierend auf dem Namen und der Beschreibung der Klasse, ob sie mit einer anderen Klasse zusammengeführt werden kann oder ob es sich tatsächlich um eine Beziehung zwischen einer Klasse und einer anderen handelt. Dies ist insbesondere für Klassen relevant, die in der Form 'hat etwas' benannt sind.
Alle Ontologiekomponenten müssen in deutscher Sprache sein!
Begriffe wie 'Bereich' oder 'Domäne' sind nicht erlaubt!
Klassen sind immer im Singular zu verwenden, auch wenn sie in der Frage im Plural stehen.

Deien Antwort muss in JSON-Format sein, ohne zusätzliche Erklärungen oder Kommentare.
"""

In [8]:
def build_cluster_ontology_prompt(base_prompt: str, questions: list[str], max_length: int = 14000) -> str:
    question_intro = "\n\nDiese Fragen sollen verarbeitet werden:\n"
    formatted_questions = "\n".join(f"{i+1}. {q.strip()}" for i, q in enumerate(questions))

    # Trim to max_length if needed
    trimmed_prompt = base_prompt.strip()
    max_question_space = max_length - len(trimmed_prompt) - len(question_intro)

    current_length = 0
    selected_questions = []
    for q in questions:
        q_line = f"{len(selected_questions)+1}. {q.strip()}\n"
        if current_length + len(q_line) > max_question_space:
            break
        selected_questions.append(q_line)
        current_length += len(q_line)

    final_prompt = f"{trimmed_prompt}{question_intro}{''.join(selected_questions)}"
    return final_prompt


def batch_questions(questions, batch_size):
    for i in range(0, len(questions), batch_size):
        yield questions[i:i + batch_size]



def call_mistral(prompt: str, model: str, max_retries: int = 5, base_delay: float = 5.0) -> dict:
    for attempt in range(max_retries):
        try:
            response = client.chat.complete(
                model=model,
                random_seed=SEED,
                messages=[{"role": "user", "content": prompt}],
                temperature=TEMPERATURE,
            )

            if response and response.choices and response.choices[0].message:
                return response
            else:
                print(f"⚠️ Received empty or malformed response: {response}")
        
        except Exception as e:
            # Check if rate limit error
            print(f"⚠️ Exception on attempt {attempt + 1}: {e}")
            if "429" in str(e):
                wait_time = base_delay * (2 ** attempt) + random.uniform(0, 2)
                print(f"⏳ Rate limit hit. Waiting {wait_time:.1f} seconds before retrying...")
                time.sleep(wait_time)
            else:
                print(f"⚠️ Error on attempt {attempt + 1}: {e}")
                time.sleep(base_delay)

    print("❌ All retries failed for prompt.")
    return None


def save_ontology(cluster, batch, ontology_str, output_dir="ontology_output"):
    cluster_id = cluster.replace(" ", "_")
    # Create filename like: Cluster_3_batch_0.json
    filename = f"onto_batch/{cluster_id}_batch_{batch}.json"
    filepath = os.path.join(output_dir, filename)

    try:
        ontology_dict = json.loads(ontology_str)
    except json.JSONDecodeError as e:
        error_path = os.path.join(output_dir, f"onto_batch/invalid_output_{cluster_id}_batch_{batch}.txt")
        print(f"❌ JSON decode error in cluster '{cluster}', batch {batch}: {e}")
        with open(error_path, "w", encoding="utf-8") as f:
            f.write(ontology_str)
            return

    # Prepare data structure with metadata
    output = {
        "cluster": cluster,
        "batch": batch,
        "ontology": {
            "Klassen": ontology_dict.get("Klassen", []),
            "Eigenschaften": ontology_dict.get("Eigenschaften", []),
            "Beziehungen": ontology_dict.get("Beziehungen", [])
        }
    }

    # Save to file
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=4)

    print(f"✅ Saved: {filepath}")



In [9]:
for cluster_name, cqs in data.items():
    if cluster_name == "Cluster -1":  # skip noise
        continue

    batches = list(batch_questions(cqs, batch_size=40))
    cluster_ontologies = []

    for i, batch in enumerate(batches):
        question_texts = [cq["cq"] for cq in batch]
        prompt = build_cluster_ontology_prompt(base_system_prompt, question_texts)
        response = call_mistral(prompt, model=model)

        output = response.choices[0].message.content.strip()
        
        save_ontology(cluster_name, i, output)
        print(f"Batch {i} for cluster '{cluster_name}' processed.")
        time.sleep(0.1)

✅ Saved: ontology_output\onto_batch/Cluster_0_batch_0.json
Batch 0 for cluster 'Cluster 0' processed.
✅ Saved: ontology_output\onto_batch/Cluster_1_batch_0.json
Batch 0 for cluster 'Cluster 1' processed.
✅ Saved: ontology_output\onto_batch/Cluster_2_batch_0.json
Batch 0 for cluster 'Cluster 2' processed.
✅ Saved: ontology_output\onto_batch/Cluster_3_batch_0.json
Batch 0 for cluster 'Cluster 3' processed.
✅ Saved: ontology_output\onto_batch/Cluster_4_batch_0.json
Batch 0 for cluster 'Cluster 4' processed.
✅ Saved: ontology_output\onto_batch/Cluster_5_batch_0.json
Batch 0 for cluster 'Cluster 5' processed.
✅ Saved: ontology_output\onto_batch/Cluster_6_batch_0.json
Batch 0 for cluster 'Cluster 6' processed.
✅ Saved: ontology_output\onto_batch/Cluster_7_batch_0.json
Batch 0 for cluster 'Cluster 7' processed.
✅ Saved: ontology_output\onto_batch/Cluster_8_batch_0.json
Batch 0 for cluster 'Cluster 8' processed.
✅ Saved: ontology_output\onto_batch/Cluster_9_batch_0.json
Batch 0 for cluster 'Cl

In [10]:
all_ontologies = []

for cluster_name, cqs in data.items():
    if cluster_name == "Cluster -1":  # skip noise
        continue

    batches = list(batch_questions(cqs, batch_size=40))
    cluster_ontologies = []
    for i, batch in enumerate(batches):
        question_texts = [cq["cq"] for cq in batch]
        prompt = build_cluster_ontology_prompt(base_system_prompt, question_texts)
        response = call_mistral(prompt, model=model)

        if not response:
            print(f"⚠️ Skipping batch {i} for cluster '{cluster_name}' due to failed response.")
            continue

        output = response.choices[0].message.content.strip()
        
        try:
            ontology_dict = json.loads(output)
            cluster_ontologies.append({
                "batch": i,
                "ontology": {
                    "Klassen": ontology_dict.get("Klassen", []),
                    "Eigenschaften": ontology_dict.get("Eigenschaften", []),
                    "Beziehungen": ontology_dict.get("Beziehungen", [])
                }
            })
        except json.JSONDecodeError as e:
            cluster_id = cluster_name.replace(" ", "_")
            error_path = os.path.join("ontology_output", f"onto_batch/invalid_output_{cluster_id}_batch_{i}.txt")
            os.makedirs(os.path.dirname(error_path), exist_ok=True)
            with open(error_path, "w", encoding="utf-8") as f:
                f.write(output)
            print(f"❌ JSON decode error in cluster '{cluster_name}', batch {i}: {e}")
            continue

        print(f"Batch {i} for cluster '{cluster_name}' processed.")
        time.sleep(0.1)

    # Append full cluster output to final list
    all_ontologies.append({
        "cluster": cluster_name,
        "batches": cluster_ontologies
    })



Batch 0 for cluster 'Cluster 0' processed.
Batch 0 for cluster 'Cluster 1' processed.
Batch 0 for cluster 'Cluster 2' processed.
Batch 0 for cluster 'Cluster 3' processed.
Batch 0 for cluster 'Cluster 4' processed.
Batch 0 for cluster 'Cluster 5' processed.
Batch 0 for cluster 'Cluster 6' processed.
Batch 0 for cluster 'Cluster 7' processed.
Batch 0 for cluster 'Cluster 8' processed.
Batch 0 for cluster 'Cluster 9' processed.
Batch 0 for cluster 'Cluster 10' processed.
Batch 1 for cluster 'Cluster 10' processed.
Batch 0 for cluster 'Cluster 11' processed.
Batch 0 for cluster 'Cluster 12' processed.
Batch 0 for cluster 'Cluster 13' processed.
Batch 0 for cluster 'Cluster 14' processed.
Batch 0 for cluster 'Cluster 15' processed.
Batch 0 for cluster 'Cluster 16' processed.
Batch 0 for cluster 'Cluster 17' processed.
Batch 0 for cluster 'Cluster 18' processed.
⚠️ Exception on attempt 1: API error occurred: Status 429
{"message":"Requests rate limit exceeded"}
⏳ Rate limit hit. Waiting 6.1

In [11]:
# Save all ontologies to a single file
merged_clusters = []

for cluster in all_ontologies:
    cluster_name = cluster["cluster"]
    merged_ontology = {
        "Klassen": set(),
        "Eigenschaften": set(),
        "Beziehungen": set()
    }

    for batch in cluster.get("batches", []):
        ontology = batch.get("ontology", {})
        merged_ontology["Klassen"].update(ontology.get("Klassen", []))
        merged_ontology["Eigenschaften"].update(ontology.get("Eigenschaften", []))
        for relation in ontology.get("Beziehungen", []):
            merged_ontology["Beziehungen"].add(tuple(relation))

    cleaned_ontology = {
        "cluster": cluster_name,
        "ontology": {
            "Klassen": sorted(merged_ontology["Klassen"]),
            "Eigenschaften": sorted(merged_ontology["Eigenschaften"]),
            "Beziehungen": [list(r) for r in sorted(merged_ontology["Beziehungen"])]
        }
    }
    merged_clusters.append(cleaned_ontology)


with open(f"{output_file_path}", "w", encoding="utf-8") as f:
    json.dump(merged_clusters, f, ensure_ascii=False, indent=4)

In [ ]:
onto = get_ontology("http://example.org/juristische_ontologie.owl")

with onto:
    class_map = {}

    for cluster in merged_clusters:
        ont = cluster["ontology"]

        # 1. Create classes
        for cls_name in ont["Klassen"]:
            owl_cls = types.new_class(cls_name, (Thing,))
            class_map[cls_name] = owl_cls

        # 2. Create properties
        for prop_name in ont["Eigenschaften"]:
            prop = types.new_class(prop_name, (ObjectProperty,))
            prop.domain = [Thing]
            prop.range = [Thing]
        # 3. Add relationships (axioms)
        for relation in ont["Beziehungen"]:
            if not isinstance(relation, list) or len(relation) != 3:
                print(f"⚠️ Invalid relation format in cluster '{cluster['cluster']}': {relation}")
                continue

            subj, pred, obj = relation

            try:
                prop = types.new_class(pred, (ObjectProperty,))
            except RuntimeError:
                prop = onto[pred]

            prop.domain = [class_map.get(subj, Thing)]
            prop.range = [class_map.get(obj, Thing)]
# Save as OWL/RDF/XML
onto.save(file="juristische_ontologie.owl", format="rdfxml")


Hi 1
Hi 2
Hi 3
Hi 3
Hi 3
Hi 3
Hi 3
Hi 3
Hi 3
Hi 4
Hi 4
Hi 4
Hi 4
Hi 4
Hi 4
Hi 4
Hi 4.5
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 2
Hi 3
Hi 3
Hi 3
Hi 4
Hi 4
Hi 4
Hi 4
Hi 4
Hi 4
Hi 4
Hi 4
Hi 4.5
⚠️ Invalid relation format in cluster 'Cluster 1': ['Beklagte', 'argumentieren', 'Anwendung', 'angegriffenes Patent']
Hi 4.6
⚠️ Invalid relation format in cluster 'Cluster 1': ['Beklagte', 'argumentieren', 'Nutzung', 'angegriffenes Patent']
Hi 4.6
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 2
Hi 3
Hi 3
Hi 3
Hi 3
Hi 3
Hi 4
Hi 4
Hi 4
Hi 4
Hi 4
Hi 4
Hi 4.5
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 4.7
Hi 4.8
Hi 5
Hi 7
Hi 2
Hi 3
Hi 3
Hi 3
Hi 3
Hi 3
Hi 3
Hi 3
Hi 4
Hi 4
H